# SEIS 766: Vision AI (SP26)
## Final Project: Latent Space Traversal
Dante Razo, razo3843@stthomas.edu

## Configuring Environment for GPU Acceleration

In [110]:
from torch import device as torch_device
from torch import cuda

# select device and verify CUDA visibility
device: torch_device = torch_device(device="cuda" if cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA not detected!")

# log hardware info
print(
    f"Using {str(object=device).upper()} on {cuda.get_device_name(device=cuda.current_device())}"
)

Using CUDA on NVIDIA GeForce RTX 5090


In [111]:
from os import environ

# configure keras
environ["KERAS_BACKEND"] = "torch"
environ["KERAS_TORCH_DEVICE"] = "cuda"

# reduce verbosity
enable_cuda_debug_blocking: bool = False
if enable_cuda_debug_blocking:
    environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [112]:
# check driver and GPU status
!uname -a && echo
!nvidia-smi

Linux ovedur 6.6.87.2-microsoft-standard-WSL2 #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux

Sat Apr 25 16:44:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.04              Driver Version: 596.21         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0A:00.0  On |                  N/A |
|  0%   49C    P0             79W /  460W |    3591MiB /  32607MiB |      2%      Default |
|         

In [113]:
from keras import backend

# configure keras for torch support
backend.set_image_data_format(data_format="channels_last")
backend.set_floatx(value="float32")
backend.clear_session()

In [114]:
from keras.mixed_precision import Policy, set_global_policy

# set global precision policy for torch
policy: Policy = Policy(name="float32")
set_global_policy(policy=policy)

# verify global data types / policies
print(f"Compute Data Type: {policy.compute_dtype}")
print(f"Variable Data Type: {policy.variable_dtype}")

Compute Data Type: float32
Variable Data Type: float32


In [115]:
from keras import config as keras_config

# configure keras
print(f"Keras Backend: {str(object=keras_config.backend()).capitalize()}")

Keras Backend: Torch


## Loading Convolutional Neural Networks

In [116]:
from keras.models import Sequential, load_model

# load model into memory
cnn1: Sequential = load_model(
    filepath="CNN/melanoma_cnn_arch1_35epochs.keras", compile=False
)  # type: ignore

# adjust metadata
cnn1.name = "cnn1-shallow"
cnn1.get_layer(name="dense").name = "embedding"
cnn1.get_layer(name="dense_1").name = "output"

# preview architecture
cnn1.summary()

Model: "cnn1-shallow"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,841 (108.75 KB)

 Trainable params: 27,841 (108.75 KB)

 Non-trainable params: 0 (0.00 B)

In [117]:
from keras.models import Sequential, load_model

# load model into memory
cnn2: Sequential = load_model(
    filepath="CNN/melanoma_cnn_arch2_35epochs.keras", compile=False
)  # type: ignore

# adjust metadata
cnn2.name = "cnn2-deep"
cnn2.get_layer(name="dense_2").name = "embedding"
cnn2.get_layer(name="dense_3").name = "output"

# preview architecture
cnn2.summary()

Model: "cnn2-deep"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,889 (429.25 KB)

 Trainable params: 109,889 (429.25 KB)

 Non-trainable params: 0 (0.00 B)

## Loading Data

In [118]:
from pandas import DataFrame, read_csv

# load ground truth metadata + classes
metadata_df: DataFrame = read_csv(
    filepath_or_buffer="../data/dante/train-ground-truth/ISIC_2020_Training_GroundTruth_v2.csv",
    usecols=["image_name", "target"],
)

# preview df
metadata_df

,image_name,target
0,ISIC_2637011,0
1,ISIC_0015719,0
2,ISIC_0052212,0
3,ISIC_0068279,0
4,ISIC_0074268,0
...,...,...
33121,ISIC_9999134,0
33122,ISIC_9999320,0
33123,ISIC_9999515,0
33124,ISIC_9999666,0


In [ ]:
from pandas.core.frame import DataFrame

# define data directory
image_root: str = "../data/dante/train/jpg"


# TODO: simplify
# derive new columns
metadata_df["file_name"] = metadata_df["image_name"].map(
    func=lambda image_name: f"{image_name}.jpg"
)
metadata_df["file_path"] = metadata_df["file_name"].map(
    func=lambda file_name: f"{image_root}/{file_name}"
)

# preview metadata_df
metadata_df

,image_name,target,file_name,file_path
0,ISIC_2637011,0,ISIC_2637011.jpg,../data/dante/train/jpg/ISIC_2637011.jpg
1,ISIC_0015719,0,ISIC_0015719.jpg,../data/dante/train/jpg/ISIC_0015719.jpg
2,ISIC_0052212,0,ISIC_0052212.jpg,../data/dante/train/jpg/ISIC_0052212.jpg
3,ISIC_0068279,0,ISIC_0068279.jpg,../data/dante/train/jpg/ISIC_0068279.jpg
4,ISIC_0074268,0,ISIC_0074268.jpg,../data/dante/train/jpg/ISIC_0074268.jpg
...,...,...,...,...
33121,ISIC_9999134,0,ISIC_9999134.jpg,../data/dante/train/jpg/ISIC_9999134.jpg
33122,ISIC_9999320,0,ISIC_9999320.jpg,../data/dante/train/jpg/ISIC_9999320.jpg
33123,ISIC_9999515,0,ISIC_9999515.jpg,../data/dante/train/jpg/ISIC_9999515.jpg
33124,ISIC_9999666,0,ISIC_9999666.jpg,../data/dante/train/jpg/ISIC_9999666.jpg


In [ ]:
# preview class distribution
metadata_df["target"].value_counts()

target
0    32542
1      584
Name: count, dtype: int64

In [120]:
# filter to only images that exist on disk
filtered_df: DataFrame = metadata_df.loc[metadata_df["target"] == 1]

# print sample size reduction
reduced_pct: float = (1 - len(filtered_df) / len(metadata_df)) * 100
print(
    f"Sample size reduced by {reduced_pct:.2f}% "
    f"({len(metadata_df)} -> {len(filtered_df)}) after filtering!"
)

# preview filtered data
filtered_df

Sample size reduced by 98.24% (33126 -> 584) after filtering!


,image_name,target,file_name,file_path
91,ISIC_0149568,1,ISIC_0149568.jpg,../data/dante/train/jpg/ISIC_0149568.jpg
235,ISIC_0188432,1,ISIC_0188432.jpg,../data/dante/train/jpg/ISIC_0188432.jpg
314,ISIC_0207268,1,ISIC_0207268.jpg,../data/dante/train/jpg/ISIC_0207268.jpg
399,ISIC_0232101,1,ISIC_0232101.jpg,../data/dante/train/jpg/ISIC_0232101.jpg
459,ISIC_0247330,1,ISIC_0247330.jpg,../data/dante/train/jpg/ISIC_0247330.jpg
...,...,...,...,...
32969,ISIC_9955163,1,ISIC_9955163.jpg,../data/dante/train/jpg/ISIC_9955163.jpg
33000,ISIC_9963177,1,ISIC_9963177.jpg,../data/dante/train/jpg/ISIC_9963177.jpg
33014,ISIC_9967383,1,ISIC_9967383.jpg,../data/dante/train/jpg/ISIC_9967383.jpg
33050,ISIC_9978107,1,ISIC_9978107.jpg,../data/dante/train/jpg/ISIC_9978107.jpg


In [121]:
def filter_by_class(working_df: DataFrame, target: int) -> DataFrame:
    """Helper function to filter a DataFrame by the given class."""

    return working_df[working_df["target"] == target]

In [122]:
from PIL import Image
from numpy import asarray, float32, stack, ndarray

# derive image dimensions from model input shape
img_height: int = cnn1.input_shape[1]
img_width: int = cnn1.input_shape[2]


# define preprocessing function
def load_image_batch(paths: list[str]) -> ndarray:
    # init empty list
    batch: list[ndarray] = []
    for path in paths:
        # load image into memory
        img: Image.Image = (
            Image.open(fp=path).convert(mode="RGB").resize(size=(img_width, img_height))
        )

        # normalize pixel values
        x: ndarray = asarray(img, dtype=float32) / 255.0

        # append to batch list
        batch.append(x)

    # return image batch
    return stack(arrays=batch, axis=0)

In [123]:
from keras import Model

# instantiate encoder for first CNN
cnn1_encoder = Model(
    inputs=cnn1.inputs,
    outputs=cnn1.get_layer(name="embedding").output,
    name="cnn1_encoder",
)

# preview architecture
cnn1_encoder.summary()

Model: "cnn1_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 128)            │         8,320 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,712 (108.25 KB)

 Trainable params: 27,712 (108.25 KB)

 Non-trainable params: 0 (0.00 B)

In [124]:
# instantiate encoder for second CNN
cnn2_encoder = Model(
    inputs=cnn2.inputs,
    outputs=cnn2.get_layer(name="embedding").output,
    name="cnn2_encoder",
)

# preview architecture
cnn2_encoder.summary()

Model: "cnn2_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 128)            │        16,512 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,760 (428.75 KB)

 Trainable params: 109,760 (428.75 KB)

 Non-trainable params: 0 (0.00 B)

## Initial Exploration of Latent Space

In [125]:
# prepare a small sample of real images for first CNN
sample_count: int = min(16, len(metadata_df))
cnn1_sample_df = metadata_df.iloc[:sample_count].copy()

cnn1_sample_paths: list[str] = cnn1_sample_df["filename"].tolist()
cnn1_sample_names: list[str] = cnn1_sample_df["image_name"].tolist()
cnn1_sample_targets = cnn1_sample_df["target"].to_numpy()

# encode images with first CNN encoder
x1_batch = load_image_batch(paths=cnn1_sample_paths)
z1_batch = cnn1_encoder.predict(x=x1_batch, verbose=0)

# store latent vectors with metadata
cnn1_latent_df: DataFrame = DataFrame(data=z1_batch, index=cnn1_sample_names)
cnn1_latent_df.index.name = "image_name"
cnn1_latent_df.columns = [f"z{i:03d}" for i in range(cnn1_latent_df.shape[1])]

cnn1_latent_df.insert(loc=0, column="target", value=cnn1_sample_targets)
cnn1_latent_df.insert(loc=1, column="filename", value=cnn1_sample_paths)

cnn1_latent_df.head()

KeyError: 'filename'

In [ ]:
# prepare a small sample of real images for first CNN
sample_count: int = min(16, len(metadata_df))
cnn1_sample_df = metadata_df.iloc[:sample_count].copy()

cnn1_sample_paths: list[str] = cnn1_sample_df["filename"].tolist()
cnn1_sample_names: list[str] = cnn1_sample_df["image_name"].tolist()
cnn1_sample_targets = cnn1_sample_df["target"].to_numpy()

# encode images with first CNN encoder
x1_batch = load_image_batch(paths=cnn1_sample_paths)
z1_batch = cnn1_encoder.predict(x=x1_batch, verbose=0)

# store latent vectors with metadata
cnn1_latent_df: DataFrame = DataFrame(data=z1_batch, index=cnn1_sample_names)
cnn1_latent_df.index.name = "image_name"
cnn1_latent_df.columns = [f"z{i:03d}" for i in range(cnn1_latent_df.shape[1])]

cnn1_latent_df.insert(loc=0, column="target", value=cnn1_sample_targets)
cnn1_latent_df.insert(loc=1, column="filename", value=cnn1_sample_paths)

cnn1_latent_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'ISIC_0149568.jpg'